In [46]:
import pandas as pd
import numpy as np
import plotly.express as pxW
import plotly.graph_objects as go
from pycountry import countries
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import plotly.express as px

In [47]:
nr = pd.read_csv(r"..\MASTER\NaturalResource.csv", index_col=0)
nr = nr[nr["Year"]==1995]

In [48]:
include_list = ['AGO', 'ARE', 'AZE', 'BFA', 'BHR', 'BOL', 'CHL', 'CIV', 'CMR',
       'COD', 'COG', 'DZA', 'ECU', 'EGY', 'ETH', 'GAB', 'GHA', 'GIN',
       'GNQ', 'IDN', 'IRN', 'IRQ', 'KAZ', 'KEN', 'KWT', 'LAO', 'LBR',
       'LBY', 'MDG', 'MLI', 'MMR', 'MNG', 'MOZ', 'MWI', 'MYS', 'NER',
       'NGA', 'OMN', 'PNG', 'QAT', 'RUS', 'RWA', 'SAU', 'TCD', 'TGO',
       'TTO', 'TZA', 'UGA', 'UZB', 'VEN', 'VNM', 'YEM', 'ZMB', 'ZWE']

In [49]:
nrs = nr[(nr["Country Code"].isin(include_list)) & (nr["Year"]==1995)] # & (nr["Year"]==2019)
nrs

,Year,Resource,Source,Price,Production,Reserves,Production_TotalValue,Reserves_TotalValue,Resource Category,Country Name,Country Code,Population
Country,,,,,,,,,,,,
Algeria,1995,Lead,Original,9.980000e+02,9.000000e+02,0.000000e+00,8.982000e+05,0.000000e+00,Subsoil Metals,Algeria,DZA,28470191
Algeria,1995,Natural Gas,Original,1.962596e+00,2.137353e+09,3.429155e+09,4.194760e+09,6.730047e+09,Hydrocarbons,Algeria,DZA,28470191
Algeria,1995,Oil,Original,1.980052e+01,4.779809e+08,3.642335e+06,9.464269e+09,7.212011e+07,Hydrocarbons,Algeria,DZA,28470191
Algeria,1995,Silver,Original,1.771000e+05,2.000000e+00,0.000000e+00,3.542000e+05,0.000000e+00,Precious Metals,Algeria,DZA,28470191
Algeria,1995,Zinc,Original,1.320000e+03,3.600000e+03,0.000000e+00,4.752000e+06,0.000000e+00,Subsoil Metals,Algeria,DZA,28470191
...,...,...,...,...,...,...,...,...,...,...,...,...
Zimbabwe,1995,Coal,Original,4.581266e+01,5.540000e+06,0.000000e+00,2.538021e+08,0.000000e+00,Hydrocarbons,Zimbabwe,ZWE,10974599
Zimbabwe,1995,Copper,Original,3.260000e+03,8.045000e+03,0.000000e+00,2.622670e+07,0.000000e+00,Subsoil Metals,Zimbabwe,ZWE,10974599
Zimbabwe,1995,Gold,Original,1.330000e+07,2.395900e+01,0.000000e+00,3.186547e+08,0.000000e+00,Precious Metals,Zimbabwe,ZWE,10974599


In [50]:
df_pivot = nrs.pivot_table(
    index=['Country', 'Country Code', 'Year', 'Population'],      # Rows
    columns='Resource',              # Columns (each resource becomes a column)
    values='Production_TotalValue' # Values to fill
).reset_index()

resource_cols = df_pivot.columns.difference(['Country', 'Country Code', 'Year', 'Population'])
df_pivot[resource_cols] = df_pivot[resource_cols].div(df_pivot['Population'], axis=0)
df_pivot.drop(columns="Population", inplace=True)

df_pivot = df_pivot.fillna(0)
df_pivot

Resource,Country,Country Code,Year,Aluminium,Bauxite,Cadmium,Coal,Cobalt,Copper,Gold,...,Manganese,Natural Gas,Natural Graphite,Nickel,Oil,Rare Earth,Silver,Tin,Vanadium,Zinc
0,Algeria,DZA,1995,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,147.338676,0.000000,0.000000,332.427296,0.000000,0.012441,0.000000,0.000000,0.166911
1,Angola,AGO,1995,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,333.856467,0.000000,0.000000,0.000000,0.000000,0.000000
2,Azerbaijan,AZE,1995,2.891403,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,57.210307,0.000000,0.000000,174.046998,0.000000,0.000000,0.000000,0.000000,0.000000
3,Bahrain,BHR,1995,1640.504760,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,921.014048,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,Bolivia,BOL,1995,0.000000,0.000000,0.000000,0.000000,0.000000,0.052576,24.329445,...,0.000000,26.364232,0.000000,0.000000,0.000000,0.000000,9.559363,15.459187,0.000000,24.495347
5,Burkina Faso,BFA,1995,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.742735,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
6,Cameroon,CMR,1995,12.266784,0.000000,0.000000,0.000000,0.000000,0.000000,0.570356,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
7,Chad,TCD,1995,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
8,Chile,CHL,1995,0.000000,0.000000,0.000000,4.682586,0.000000,558.451382,40.818128,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,12.691789,0.000000,0.000000,3.216819
9,"Congo, Dem. Rep.",COD,1995,0.000000,0.000000,0.000000,0.095886,2.345247,2.564789,0.353200,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.881117,0.000000,0.134217


In [70]:
import plotly.express as px

# =============================================================================
# STEP 1: Prepare data (get latest year per country)
# =============================================================================

df_latest = df_pivot.sort_values('Year', ascending=True).groupby(['Country', 'Country Code']).first().reset_index()
feature_cols = [c for c in df_latest.columns if c not in ['Country', 'Country Code', 'Year']]

# =============================================================================
# STEP 2: Log transform to reduce outlier influence
# =============================================================================

X = df_latest[feature_cols].fillna(0)
X_log = np.log1p(X)
X_scaled = X_log

# =============================================================================
# STEP 4: PCA - Dimensionality Reduction
# =============================================================================

pca = PCA(n_components=2)
pca_components = pca.fit_transform(X_scaled)

print(f"\nPCA Explained Variance:")
for i, var in enumerate(pca.explained_variance_ratio_):
    cumulative = pca.explained_variance_ratio_[:i+1].sum()
    print(f"  PC{i+1}: {var*100:.1f}% (cumulative: {cumulative*100:.1f}%)")

n_components_for_clustering = 2
X_pca = pca_components[:, :n_components_for_clustering]

print(f"\nUsing {n_components_for_clustering} PCs for clustering")
print(f"  Explains {pca.explained_variance_ratio_[:n_components_for_clustering].sum()*100:.1f}% of variance")

# =============================================================================
# STEP 5: K-Means Clustering on PCA Components
# =============================================================================

n_clusters = 4
kmeans = KMeans(n_clusters=n_clusters, n_init=10, random_state=42)
clusters = kmeans.fit_predict(X_pca)

# =============================================================================
# STEP 6: Create results dataframe + stable data-driven labels
# =============================================================================

pca_df = pd.DataFrame({
    'Country': df_latest['Country'],
    'Country Code': df_latest['Country Code'],
    'Year': df_latest['Year'],
    'PC1': pca_components[:, 0],
    'PC2': pca_components[:, 1],
    'Cluster_num': clusters
})

# Attach log-transformed features to compute cluster means
pca_df_with_features = pca_df.copy()
pca_df_with_features[feature_cols] = X_log.values

# Identify oil and mineral columns by name
oil_cols     = [c for c in feature_cols if any(k in c.lower() for k in ['oil', 'petroleum', 'crude'])]
mineral_cols = [c for c in feature_cols if any(k in c.lower() for k in ['mineral', 'coal', 'metal', 'copper', 'gold', 'iron', 'zinc'])]

print(f"\nOil columns found:     {oil_cols}")
print(f"Mineral columns found: {mineral_cols}")

# Compute per-cluster means
cluster_means = pca_df_with_features.groupby('Cluster_num')[oil_cols + mineral_cols].mean()
cluster_means['oil_mean']     = cluster_means[oil_cols].mean(axis=1)     if oil_cols     else 0
cluster_means['mineral_mean'] = cluster_means[mineral_cols].mean(axis=1) if mineral_cols else 0

oil_threshold     = cluster_means['oil_mean'].median()
mineral_threshold = cluster_means['mineral_mean'].median()

def assign_label(row):
    high_oil     = row['oil_mean']     > oil_threshold
    high_mineral = row['mineral_mean'] > mineral_threshold
    if high_oil and high_mineral:
        return 'Oil, some minerals'
    elif high_oil:
        return 'Oil, no minerals'
    elif high_mineral:
        return 'No oil, minerals'
    else:
        return 'No oil, no minerals'

label_map = {idx: assign_label(row) for idx, row in cluster_means.iterrows()}
pca_df['Cluster'] = pca_df['Cluster_num'].map(label_map)

print("\nCluster label assignments:")
print(cluster_means[['oil_mean', 'mineral_mean']].assign(Label=cluster_means.index.map(label_map)))

# =============================================================================
# STEP 7: Calculate PCA loadings for biplot arrows
# =============================================================================

loadings = pca.components_.T * np.sqrt(pca.explained_variance_)
loadings_df = pd.DataFrame(
    loadings[:, :2],
    columns=['PC1', 'PC2'],
    index=feature_cols
)

scale_factor = 2.5
loadings_df_scaled = loadings_df * scale_factor

top_n = 15
loading_importance = loadings_df.abs().sum(axis=1)
top_features = loading_importance.nlargest(top_n).index

print(f"\nTop {top_n} most influential resources:")
for feat in top_features:
    print(f"  {feat}: {loading_importance[feat]:.3f}")

# =============================================================================
# STEP 8: Build stable color map (alphabetical = Plotly's default Bold order)
# then use color_discrete_map so all charts stay in sync
# =============================================================================

_bold = px.colors.qualitative.Bold
_labels = sorted(pca_df['Cluster'].unique())  # alphabetical order
label_to_color = {label: _bold[i % len(_bold)] for i, label in enumerate(_labels)}

# cluster_names exposed for downstream cells (map, Rosling, etc.)
cluster_names = {row['Cluster_num']: row['Cluster']
                 for _, row in pca_df[['Cluster_num', 'Cluster']].drop_duplicates().iterrows()}

print("\nColor assignments:")
for label, color in label_to_color.items():
    print(f"  {label}: {color}")

# =============================================================================
# STEP 9: Create the biplot
# =============================================================================

fig = px.scatter(
    pca_df,
    x='PC1',
    y='PC2',
    color='Cluster',
    hover_data=['Country', 'Country Code', 'Year'],
    title=f'K-Means Clusters on Natural Resources (k={n_clusters})',
    color_discrete_map=label_to_color  # explicit map = stable colors across all charts
)

for feature in top_features:
    fig.add_annotation(
        x=loadings_df_scaled.loc[feature, 'PC1'],
        y=loadings_df_scaled.loc[feature, 'PC2'],
        ax=0, ay=0,
        xref='x', yref='y',
        axref='x', ayref='y',
        showarrow=True,
        arrowhead=2,
        arrowsize=1,
        arrowwidth=2,
        arrowcolor='black'
    )
    fig.add_annotation(
        x=loadings_df_scaled.loc[feature, 'PC1'] * 1.15,
        y=loadings_df_scaled.loc[feature, 'PC2'] * 1.15,
        text=feature,
        showarrow=False,
        font=dict(size=10, color='black'),
        xref='x', yref='y'
    )

fig.update_layout(
    width=1000,
    height=700,
    xaxis_title=f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)",
    yaxis_title=f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)",
)

fig.write_html("..\\figures\\clusters\\pca_biplot.html")
fig.write_image("..\\figures\\clusters\\pca_biplot.png")
fig.show()


PCA Explained Variance:
  PC1: 53.1% (cumulative: 53.1%)
  PC2: 16.7% (cumulative: 69.8%)

Using 2 PCs for clustering
  Explains 69.8% of variance

Oil columns found:     ['Oil']
Mineral columns found: ['Coal', 'Copper', 'Gold', 'Zinc']

Cluster label assignments:
             oil_mean  mineral_mean                Label
Cluster_num                                             
0            0.000000      2.344419     No oil, minerals
1            6.730976      0.897476   Oil, some minerals
2            4.964875      0.051355     Oil, no minerals
3            0.140157      0.252047  No oil, no minerals

Top 15 most influential resources:
  Oil: 3.401
  Natural Gas: 2.711
  Copper: 1.607
  Gold: 1.317
  Coal: 1.029
  Aluminium: 0.811
  Silver: 0.486
  Zinc: 0.330
  Manganese: 0.212
  Nickel: 0.208
  Cobalt: 0.151
  Bauxite: 0.148
  Tin: 0.117
  Lead: 0.108
  Magnesium compounds: 0.053

Color assignments:
  No oil, minerals: rgb(127, 60, 141)
  No oil, no minerals: rgb(17, 165, 121)
  Oil,

In [69]:
import plotly.graph_objects as go
import plotly.express as px
import pandas as pd
import numpy as np

# =============================================================================
# CONFIGURATION
# =============================================================================

class MapConfig:
    dominance_threshold: float = 15.0
    min_share_display: float = 1.0
    use_prices: bool = True
    dominant_border_color: str = 'red'

config = MapConfig()

# =============================================================================
# SHARED COLOR + LABEL MAPPING
# label_to_color and cluster_names are defined in the biplot cell
# Just verify they're available here
# =============================================================================

assert 'label_to_color' in dir(), "Run the biplot cell first to define label_to_color"
assert 'cluster_names' in dir(), "Run the biplot cell first to define cluster_names"

# =============================================================================
# STEP 1: Get TOTAL Production Values from Original NR DataFrame
# =============================================================================

def get_total_production_values(nr):
    df = nr.reset_index() if nr.index.name == 'Country' else nr.copy()
    df_total = df.pivot_table(
        index=['Country', 'Country Code'],
        columns='Resource',
        values='Production_TotalValue',
        aggfunc='sum'
    ).reset_index()
    df_total = df_total.fillna(0)
    return df_total

df_total_prod = get_total_production_values(nr)
prod_feature_cols = [c for c in df_total_prod.columns if c not in ['Country', 'Country Code']]
print(f"Total production data: {len(df_total_prod)} countries, {len(prod_feature_cols)} resources")

# =============================================================================
# STEP 2: Calculate Global Production Shares
# =============================================================================

def calculate_shares(df, feature_cols):
    df_shares = df.copy()
    share_cols = []
    for col in feature_cols:
        total = df_shares[col].sum()
        if total > 0:
            share_col = f"{col}_Share"
            df_shares[share_col] = (df_shares[col] / total) * 100
            share_cols.append(share_col)
    return df_shares, share_cols

df_prod_shares, prod_share_cols = calculate_shares(df_total_prod, prod_feature_cols)
print(f"Production share columns: {len(prod_share_cols)}")

# =============================================================================
# STEP 3: Merge Production Data into pca_df
# =============================================================================

pca_df = pca_df.merge(df_prod_shares, on='Country Code', how='left', suffixes=('', '_prod'))
if 'Country_prod' in pca_df.columns:
    pca_df = pca_df.drop(columns=['Country_prod'])

# =============================================================================
# STEP 4: Identify Dominant Countries by PRODUCTION
# =============================================================================

def identify_dominant_countries(pca_df, share_cols, threshold=15.0):
    pca_df = pca_df.copy()
    pca_df['Dominant_Resources'] = pca_df.apply(
        lambda row: [col.replace('_Share', '') for col in share_cols
                    if col in row.index and pd.notna(row.get(col, 0)) and row.get(col, 0) >= threshold],
        axis=1
    )
    pca_df['Is_Dominant'] = pca_df['Dominant_Resources'].apply(lambda x: len(x) > 0)
    valid_cols = [c for c in share_cols if c in pca_df.columns]
    pca_df['Max_Production_Share'] = pca_df[valid_cols].max(axis=1) if valid_cols else 0
    print(f"Dominant countries (>{threshold}% of any resource's global production): {pca_df['Is_Dominant'].sum()}")
    return pca_df

pca_df = identify_dominant_countries(pca_df, prod_share_cols, config.dominance_threshold)

# =============================================================================
# STEP 5: Create Rich Hover Text
# =============================================================================

def create_hover_text(row, prod_cols, share_cols, config, cluster_names):
    cluster_label = cluster_names.get(int(row['Cluster_num']), f"Cluster {row['Cluster_num']}")
    lines = [f"<b>{row['Country']}</b>", f"Cluster: {cluster_label}"]

    if row.get('Is_Dominant', False):
        lines.append(f"<br>🔴 <b>MAJOR PRODUCER (>{int(config.dominance_threshold)}% global)</b>")
        for resource in row.get('Dominant_Resources', []):
            share_col = f"{resource}_Share"
            if share_col in row.index and pd.notna(row[share_col]):
                lines.append(f"   • {resource}: {row[share_col]:.1f}%")

    resource_values = []
    for col in prod_cols:
        val = row.get(col, 0)
        share_col = f"{col}_Share"
        share = row.get(share_col, 0) if share_col in row.index else 0
        if pd.notna(val) and val > 0:
            resource_values.append((col, val, share))

    resource_values.sort(key=lambda x: x[1], reverse=True)
    top_resources = resource_values[:5]

    if top_resources:
        lines.append("<br><b>Top Production (Total Value):</b>")
        for resource, value, share in top_resources:
            if value > 1e9:
                val_str = f"${value/1e9:.1f}B"
            elif value > 1e6:
                val_str = f"${value/1e6:.1f}M"
            elif value > 1e3:
                val_str = f"${value/1e3:.1f}K"
            else:
                val_str = f"${value:,.0f}"
            share_str = f" ({share:.1f}% global)" if share >= config.min_share_display else ""
            lines.append(f"   {resource}: {val_str}{share_str}")
    else:
        lines.append("<br>(No significant production)")

    return "<br>".join(lines)

pca_df['hover_text'] = pca_df.apply(
    lambda row: create_hover_text(row, prod_feature_cols, prod_share_cols, config, cluster_names),
    axis=1
)

# =============================================================================
# STEP 6: Create Enhanced Map
# =============================================================================

def create_enhanced_map(pca_df, config, cluster_names, label_to_color):
    df_map = pca_df[pca_df['Country Code'].notna()].copy()
    df_map['Cluster_int'] = df_map['Cluster_num'].astype(int)
    df_map['Cluster_Name'] = df_map['Cluster_int'].map(cluster_names)

    fig = go.Figure()

    # Non-dominant countries
    for cid in sorted(df_map['Cluster_int'].unique()):
        subset = df_map[(df_map['Cluster_int'] == cid) & (~df_map['Is_Dominant'])]
        cname = cluster_names.get(cid, f"Cluster {cid}")
        color = label_to_color[cname]
        if len(subset) > 0:
            fig.add_trace(go.Choropleth(
                locations=subset['Country Code'],
                z=[cid] * len(subset),
                colorscale=[[0, color], [1, color]],
                showscale=False,
                customdata=subset['hover_text'].values,
                hovertemplate="%{customdata}<extra></extra>",
                name=f"{cname} ({len(subset)})",
                marker=dict(line=dict(color='white', width=0.5)),
                legendgroup=f"cluster_{cid}",
                legendgrouptitle_text=cname
            ))

    # Dominant countries (red border, same fill color)
    for cid in sorted(df_map['Cluster_int'].unique()):
        subset = df_map[(df_map['Cluster_int'] == cid) & (df_map['Is_Dominant'])]
        cname = cluster_names.get(cid, f"Cluster {cid}")
        color = label_to_color[cname]
        if len(subset) > 0:
            fig.add_trace(go.Choropleth(
                locations=subset['Country Code'],
                z=[cid] * len(subset),
                colorscale=[[0, color], [1, color]],
                showscale=False,
                customdata=subset['hover_text'].values,
                hovertemplate="%{customdata}<extra></extra>",
                name=f"{cname} 🔴 ({len(subset)})",
                marker=dict(line=dict(color=config.dominant_border_color, width=1.5)),
                legendgroup=f"cluster_{cid}"
            ))

    fig.update_geos(
        projection_type="natural earth",
        showcountries=True, showcoastlines=True,
        countrycolor="lightgray", coastlinecolor="lightgray",
        showland=True, landcolor="whitesmoke",
        showocean=True, oceancolor="aliceblue",
        showframe=False
    )

    n_traces = len(fig.data)
    buttons = [dict(label="All Clusters", method="update", args=[{"visible": [True] * n_traces}])]
    for cid in sorted(df_map['Cluster_int'].unique()):
        cname = cluster_names.get(cid, f"Cluster {cid}")
        visibility = [trace.name.startswith(cname) for trace in fig.data]
        buttons.append(dict(label=cname, method="update", args=[{"visible": visibility}]))
    visibility_dominant = ['🔴' in trace.name for trace in fig.data]
    buttons.append(dict(label="🔴 Major Producers Only", method="update", args=[{"visible": visibility_dominant}]))

    method = "Total Production Value" if config.use_prices else "Total Production Volume"
    threshold = int(config.dominance_threshold)

    fig.update_layout(
        title=dict(
            text=(f"Natural Resource Clusters by {method} per capita<br>"
                  f"<sup>🔴 Red border = Major Producer (>{threshold}% of a resource's global production)</sup>"),
            x=0.45, xanchor='center', font=dict(size=14)
        ),
        width=1100, height=550,
        margin=dict(l=10, r=150, t=60, b=10),
        legend=dict(
            x=1.01, y=0.3, xanchor='left', yanchor='middle',
            bgcolor="rgba(255,255,255,0.95)", bordercolor="lightgray", borderwidth=1,
            title=dict(text="<b>Clusters</b><br><sup>(click to toggle)</sup>"),
            font=dict(size=10), itemclick="toggle", itemdoubleclick="toggleothers",
            tracegroupgap=5
        ),
        updatemenus=[dict(
            type="dropdown", direction="down",
            x=1.01, y=0.95, xanchor="left", yanchor="top",
            bgcolor="white", bordercolor="lightgray", borderwidth=1,
            buttons=buttons, showactive=True, active=0, font=dict(size=10)
        )],
        annotations=[dict(
            text="<b>Filter:</b>", x=1.01, y=1.0,
            xref="paper", yref="paper",
            showarrow=False, font=dict(size=11), xanchor="left"
        )],
        geo=dict(center=dict(lat=20, lon=0), projection_scale=1.1)
    )
    return fig

fig_map = create_enhanced_map(pca_df, config, cluster_names, label_to_color)

# =============================================================================
# STEP 7: Display and Export
# =============================================================================

fig_map.show()

fig_map.write_image(r"..\\figures\\clusters\\clusters_map.png")
fig_map.write_html(
    r"..\\figures\\clusters\\clusters_map.html",
    include_plotlyjs=True, full_html=True,
    config={
        'displayModeBar': True,
        'toImageButtonOptions': {
            'format': 'png', 'filename': 'resource_clusters_map',
            'height': 550, 'width': 1100, 'scale': 2
        }
    }
)
print("✅ Map exported to: resource_clusters_map.html")

# =============================================================================
# STEP 8: Print Cluster Summary
# =============================================================================

print("\n" + "="*70)
print("CLUSTER SUMMARY")
print("="*70)

for cid, cname in sorted(cluster_names.items()):
    cluster_df = pca_df[pca_df['Cluster_num'] == cid]
    countries = sorted(cluster_df['Country'].tolist())
    dominant = cluster_df[cluster_df['Is_Dominant']]['Country'].tolist()
    print(f"\n{cname} ({len(countries)} countries):")
    print(f"  {', '.join(countries[:10])}{'...' if len(countries) > 10 else ''}")
    if dominant:
        print(f"  🔴 Major Producers: {', '.join(dominant)}")

Total production data: 137 countries, 20 resources
Production share columns: 20
Dominant countries (>15.0% of any resource's global production): 4


✅ Map exported to: resource_clusters_map.html

CLUSTER SUMMARY

No oil, minerals (6 countries):
  Bolivia, Chile, Mongolia, Papua New Guinea, Zambia, Zimbabwe
  🔴 Major Producers: Chile, Zambia

Oil, some minerals (15 countries):
  Algeria, Indonesia, Iran, Islamic Rep., Kazakhstan, Kuwait, Libya, Malaysia, Oman, Qatar, Russian Federation...
  🔴 Major Producers: Indonesia, Russian Federation

Oil, no minerals (11 countries):
  Angola, Azerbaijan, Bahrain, Congo, Rep., Ecuador, Egypt, Arab Rep., Equatorial Guinea, Gabon, Iraq, Nigeria...

No oil, no minerals (20 countries):
  Burkina Faso, Cameroon, Chad, Congo, Dem. Rep., Cote d'Ivoire, Ethiopia, Ghana, Guinea, Kenya, Lao PDR...


In [53]:
# =============================================================================
# PART A: Complete Factor Loadings for All 3 PCA Components
# =============================================================================

# Get loadings (correlations between original variables and PCs)
loadings_full = pd.DataFrame(
    pca.components_.T,
    columns=['PC1', 'PC2'],
    index=feature_cols
)

# Display top loadings for each component
print("=" * 70)
print("FACTOR LOADINGS BY PRINCIPAL COMPONENT")
print("=" * 70)

for pc in ['PC1', 'PC2']:
    var_explained = pca.explained_variance_ratio_[int(pc[-1])-1] * 100
    print(f"\n{pc} ({var_explained:.1f}% variance explained)")
    print("-" * 50)
    
    # Sort by absolute loading value
    sorted_loadings = loadings_full[pc].reindex(
        loadings_full[pc].abs().sort_values(ascending=False).index
    )
    
    # Show top 10 positive and negative loadings
    print("\nTop 10 Positive Loadings:")
    top_pos = sorted_loadings[sorted_loadings > 0].head(10)
    for feat, val in top_pos.items():
        print(f"  {feat:40s} {val:+.4f}")
    
    print("\nTop 10 Negative Loadings:")
    top_neg = sorted_loadings[sorted_loadings < 0].head(10)
    for feat, val in top_neg.items():
        print(f"  {feat:40s} {val:+.4f}")

# =============================================================================
# PART B: Loadings Heatmap Visualisation
# =============================================================================

# Select top features by overall importance
top_20_features = loadings_full.abs().sum(axis=1).nlargest(20).index

fig_loadings = px.imshow(
    loadings_full.loc[top_20_features].T,
    labels=dict(x="Resource/Feature", y="Principal Component", color="Loading"),
    title="PCA Factor Loadings Heatmap (Top 20 Features)",
    color_continuous_scale='RdBu_r',
    aspect='auto',
    zmin=-1, zmax=1
)
fig_loadings.update_layout(width=1200, height=400)
fig_loadings.show()

# =============================================================================
# PART C: Cluster Profile - Average PC Scores by Cluster
# =============================================================================

print("\n" + "=" * 70)
print("CLUSTER PROFILES: AVERAGE PCA COMPONENT SCORES")
print("=" * 70)

cluster_means = pca_df.groupby('Cluster')[['PC1', 'PC2']].mean()
cluster_stds = pca_df.groupby('Cluster')[['PC1', 'PC2']].std()
cluster_counts = pca_df.groupby('Cluster').size()

print("\nMean PC Scores by Cluster:")
print(cluster_means.round(3).to_string())

print("\nCluster Sizes:")
print(cluster_counts.to_string())

# Interpretation table
print("\n" + "-" * 70)
print("CLUSTER INTERPRETATION:")
print("-" * 70)

for cluster in sorted(pca_df['Cluster'].unique()):
    n = cluster_counts[cluster]
    pc1_mean = cluster_means.loc[cluster, 'PC1']
    pc2_mean = cluster_means.loc[cluster, 'PC2']

    
    print(f"\nCluster {cluster} (n={n}):")
    print(f"  PC1: {pc1_mean:+.2f}  |  PC2: {pc2_mean:+.2f}  ")
    
    # Sample countries
    sample = pca_df[pca_df['Cluster'] == cluster]['Country'].head(5).tolist()
    print(f"  Sample countries: {', '.join(sample)}")

# =============================================================================
# PART D: Visualise Cluster Means
# =============================================================================

cluster_means_long = cluster_means.reset_index().melt(
    id_vars='Cluster',
    var_name='Component',
    value_name='Mean Score'
)

fig_cluster = px.bar(
    cluster_means_long,
    x='Cluster',
    y='Mean Score',
    color='Component',
    barmode='group',
    title='Average PCA Component Scores by Cluster',
    color_discrete_sequence=['#1f77b4', '#ff7f0e', '#2ca02c']
)
fig_cluster.update_layout(width=900, height=500)
fig_cluster.add_hline(y=0, line_dash="dash", line_color="gray")
fig_cluster.show()

# =============================================================================
# PART E: Radar Chart for Cluster Profiles
# =============================================================================

fig_radar = go.Figure()

for cluster in sorted(pca_df['Cluster'].unique()):
    values = cluster_means.loc[cluster].tolist()
    values.append(values[0])  # Close the radar
    
    fig_radar.add_trace(go.Scatterpolar(
        r=values,
        theta=['PC1', 'PC2',],
        name=f'Cluster {cluster}',
        fill='toself',
        opacity=0.6
    ))

fig_radar.update_layout(
    polar=dict(radialaxis=dict(visible=True)),
    title='Cluster Profiles: Radar Chart of Mean PC Scores',
    width=700,
    height=600
)
fig_radar.show()

FACTOR LOADINGS BY PRINCIPAL COMPONENT

PC1 (53.1% variance explained)
--------------------------------------------------

Top 10 Positive Loadings:
  Oil                                      +0.8249
  Natural Gas                              +0.5399
  Aluminium                                +0.1136
  Nickel                                   +0.0157
  Manganese                                +0.0156
  Coal                                     +0.0079
  Magnesium compounds                      +0.0044
  Vanadium                                 +0.0022
  Lead                                     +0.0020
  Cadmium                                  +0.0011

Top 10 Negative Loadings:
  Gold                                     -0.0835
  Copper                                   -0.0810
  Silver                                   -0.0249
  Cobalt                                   -0.0166
  Tin                                      -0.0140
  Lithium                                  -0.0038
  Natura


CLUSTER PROFILES: AVERAGE PCA COMPONENT SCORES

Mean PC Scores by Cluster:
                       PC1    PC2
Cluster                          
No oil, minerals    -3.740  3.471
No oil, no minerals -3.323 -1.058
Oil, no minerals     1.605 -1.457
Oil, some minerals   4.750  1.091

Cluster Sizes:
Cluster
No oil, minerals        6
No oil, no minerals    20
Oil, no minerals       11
Oil, some minerals     15

----------------------------------------------------------------------
CLUSTER INTERPRETATION:
----------------------------------------------------------------------

Cluster No oil, minerals (n=6):
  PC1: -3.74  |  PC2: +3.47  
  Sample countries: Bolivia, Chile, Mongolia, Papua New Guinea, Zambia

Cluster No oil, no minerals (n=20):
  PC1: -3.32  |  PC2: -1.06  
  Sample countries: Burkina Faso, Cameroon, Chad, Congo, Dem. Rep., Cote d'Ivoire

Cluster Oil, no minerals (n=11):
  PC1: +1.60  |  PC2: -1.46  
  Sample countries: Angola, Azerbaijan, Bahrain, Congo, Rep., Ecuador

Cluster

In [54]:
# =============================================================================
# EXPORT PCA RESULTS TO EXCEL - CONFIGURABLE BY ANALYSIS TYPE
# =============================================================================

import os

# ┌─────────────────────────────────────────────────────────────────────────────
# │ CHANGE THIS VARIABLE TO SWITCH BETWEEN ANALYSES
# └─────────────────────────────────────────────────────────────────────────────

ANALYSIS = "agg"  # Options: "1995", "2019", "agg"

# =============================================================================

output_dir = r"..\MASTER\cluster_plots_loadings\agg"

os.makedirs(output_dir, exist_ok=True)

output_file = f'{output_dir}/pca_results_{ANALYSIS}.xlsx'

# Prepare top 20 features loadings
loadings_top20 = loadings_full.loc[top_20_features]

# Save all data to single Excel with multiple sheets
with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    
    # PCA Loadings
    loadings_full.to_excel(writer, sheet_name=f'{ANALYSIS}_Loadings_Full')
    loadings_top20.to_excel(writer, sheet_name=f'{ANALYSIS}_Loadings_Top20')
    
    # Cluster Profiles
    cluster_means.to_excel(writer, sheet_name=f'{ANALYSIS}_Cluster_Means')
    cluster_stds.to_excel(writer, sheet_name=f'{ANALYSIS}_Cluster_StdDev')
    cluster_counts.to_frame('Count').to_excel(writer, sheet_name=f'{ANALYSIS}_Cluster_Counts')
    cluster_means_long.to_excel(writer, sheet_name=f'{ANALYSIS}_Means_Long', index=False)

print(f"✓ Saved: {output_file}")
print(f"  Sheets: {ANALYSIS}_Loadings_Full, {ANALYSIS}_Loadings_Top20,")
print(f"          {ANALYSIS}_Cluster_Means, {ANALYSIS}_Cluster_StdDev,")
print(f"          {ANALYSIS}_Cluster_Counts, {ANALYSIS}_Means_Long")

# -----------------------------------------------------------------------------
# Save Visualizations as HTML (with analysis name)
# -----------------------------------------------------------------------------

fig_loadings.write_html(f'{output_dir}/heatmap_{ANALYSIS}.html')
fig_cluster.write_html(f'{output_dir}/barchart_{ANALYSIS}.html')
fig_radar.write_html(f'{output_dir}/radar_{ANALYSIS}.html')
fig.write_html(f'{output_dir}/map_{ANALYSIS}.html')


print(f"\n✓ Saved HTML figures:")
print(f"  - heatmap_{ANALYSIS}.html")
print(f"  - barchart_{ANALYSIS}.html")
print(f"  - radar_{ANALYSIS}.html")
print(f"  - map_{ANALYSIS}.html")
print(f"\n{'='*50}")
print(f"Analysis '{ANALYSIS}' exported successfully!")
print(f"{'='*50}")

✓ Saved: ..\MASTER\cluster_plots_loadings\agg/pca_results_agg.xlsx
  Sheets: agg_Loadings_Full, agg_Loadings_Top20,
          agg_Cluster_Means, agg_Cluster_StdDev,
          agg_Cluster_Counts, agg_Means_Long

✓ Saved HTML figures:
  - heatmap_agg.html
  - barchart_agg.html
  - radar_agg.html
  - map_agg.html

Analysis 'agg' exported successfully!


In [55]:
pca_df.to_csv(r"..\MASTER\clustersagg.csv")

In [56]:
pca_df

,Country,Country Code,Year,PC1,PC2,Cluster_num,Cluster,Aluminium,Bauxite,Cadmium,...,Oil_Share,Rare Earth_Share,Silver_Share,Tin_Share,Vanadium_Share,Zinc_Share,Dominant_Resources,Is_Dominant,Max_Production_Share,hover_text
0,Algeria,DZA,1995,4.053870,-0.725517,1,"Oil, some minerals",0.000000e+00,0.0,0.0,...,1.962343,0.000000,0.014028,0.000000,0.000000,0.050865,[],False,2.988082,"<b>Algeria</b><br>Cluster: Oil, some minerals<..."
1,Angola,AGO,1995,1.358332,-2.241665,2,"Oil, no minerals",0.000000e+00,0.0,0.0,...,0.948334,0.000000,0.000000,0.000000,0.000000,0.000000,[],False,0.948334,"<b>Angola</b><br>Cluster: Oil, no minerals<br>..."
2,Azerbaijan,AZE,1995,3.171730,-0.722914,2,"Oil, no minerals",2.222000e+07,0.0,0.0,...,0.277326,0.000000,0.000000,0.000000,0.000000,0.000000,[],False,0.313181,"<b>Azerbaijan</b><br>Cluster: Oil, no minerals..."
3,Bahrain,BHR,1995,1.089200,1.725204,2,"Oil, no minerals",9.168437e+08,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,[],False,2.282288,"<b>Bahrain</b><br>Cluster: Oil, no minerals<br..."
4,Bolivia,BOL,1995,-2.016490,1.976807,0,"No oil, minerals",0.000000e+00,0.0,0.0,...,0.000000,0.000000,2.981290,8.115493,0.000000,2.064701,[],False,8.115493,"<b>Bolivia</b><br>Cluster: No oil, minerals<br..."
5,Burkina Faso,BFA,1995,-3.521375,-1.130380,3,"No oil, no minerals",0.000000e+00,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,[],False,0.062960,"<b>Burkina Faso</b><br>Cluster: No oil, no min..."
6,Cameroon,CMR,1995,-3.181220,-0.938168,3,"No oil, no minerals",1.601860e+08,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,[],False,0.398749,"<b>Cameroon</b><br>Cluster: No oil, no mineral..."
7,Chad,TCD,1995,-3.437162,-1.598113,3,"No oil, no minerals",0.000000e+00,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,[],False,0.000000,"<b>Chad</b><br>Cluster: No oil, no minerals<br..."
8,Chile,CHL,1995,-4.313387,5.452861,0,"No oil, minerals",0.000000e+00,0.0,0.0,...,0.000000,0.000000,7.302183,0.000000,0.000000,0.500213,"[Copper, Lithium]",True,24.485655,"<b>Chile</b><br>Cluster: No oil, minerals<br><..."
9,"Congo, Dem. Rep.",COD,1995,-3.593432,-0.563264,3,"No oil, no minerals",0.000000e+00,0.0,0.0,...,0.000000,0.000000,0.000000,2.610008,0.000000,0.063835,[],False,8.901919,"<b>Congo, Dem. Rep.</b><br>Cluster: No oil, no..."


# Rosling arrows chart

In [60]:
cluster1995 = pd.read_csv(r"..\MASTER\clustersagg.csv")
master = pd.read_csv(r"..\MASTER\Master.csv", index_col=0)

include_list = ['AGO', 'ARE', 'AZE', 'BFA', 'BHR', 'BOL', 'CHL', 'CIV', 'CMR',
       'COD', 'COG', 'DZA', 'ECU', 'EGY', 'ETH', 'GAB', 'GHA', 'GIN',
       'GNQ', 'IDN', 'IRN', 'IRQ', 'KAZ', 'KEN', 'KWT', 'LAO', 'LBR',
       'LBY', 'MDG', 'MLI', 'MMR', 'MNG', 'MOZ', 'MWI', 'MYS', 'NER',
       'NGA', 'OMN', 'PNG', 'QAT', 'RUS', 'RWA', 'SAU', 'TCD', 'TGO',
       'TTO', 'TZA', 'UGA', 'UZB', 'VEN', 'VNM', 'YEM', 'ZMB', 'ZWE']

master = master[(master["Country Code"].isin(include_list))]


master = pd.merge(master, cluster1995[["Country Code", "Cluster"]], on="Country Code", how= "left")
master.head()

,Country Code,Country Name,Year,Access to electricity (% of population),Adjusted savings: gross savings (% of GNI),Agriculture,Capital depreciation rate,Clientelism index,"Death rates, crude per 1000 people",Domestic credit to private sector (% of GDP),...,"Use of IMF credit (DOD, current US$)",Total_Production,Total_Reserves,Total_Production_Value,Total_Reserves_Value,Hydrocarbons_Dominant,Subsoil_Metals_Dominant,Precious_Metals_Dominant,Population,Cluster
0,AGO,Angola,1995,24.2,48.055414,9.386791,0.035367,0.84,18.700,22.274928,...,0.0,230991930.0,1140625.0,4.573759e+09,2.258496e+07,1,0,0,13699778.0,"Oil, no minerals"
1,AGO,Angola,1996,24.2,48.055414,9.386791,0.038398,0.84,18.445,22.274928,...,0.0,261331263.9,1348675.0,5.894739e+09,3.042149e+07,1,0,0,14170973.0,"Oil, no minerals"
2,AGO,Angola,1997,24.2,48.055414,9.386791,0.040405,0.84,18.184,22.274928,...,0.0,270465000.0,1423500.0,5.405294e+09,2.844892e+07,1,0,0,14660413.0,"Oil, no minerals"
3,AGO,Angola,1998,24.2,48.055414,9.386791,0.040825,0.84,18.925,22.274928,...,0.0,266760000.0,1470950.0,3.392030e+09,1.870410e+07,1,0,0,15159370.0,"Oil, no minerals"
4,AGO,Angola,1999,24.2,48.055414,9.386791,0.041406,0.84,18.518,22.274928,...,374707057.4,271947000.0,1843250.0,4.689445e+09,3.178494e+07,1,0,0,15667235.0,"Oil, no minerals"


In [67]:
# ============ CONFIGURATION ============
COL_COUNTRY_CODE = 'Country Code'
COL_COUNTRY_NAME = 'Country Name'
COL_YEAR = 'Year'
COL_ECI = 'Economic Complexity Index'
COL_GDP_PC = 'GDP per capita (constant prices, PPP)'
COL_PRODUCTION = 'Total_Production_Value'
COL_CLUSTER = 'Cluster'

# Use label_to_color and cluster_names from biplot cell
# cluster_names: {int -> string label}, label_to_color: {string label -> hex color}
CLUSTER_NAMES = cluster_names  # e.g. {0: 'Oil, no minerals', 1: 'No oil, no minerals', ...}
CLUSTER_COLORS = {cid: label_to_color[cname] for cid, cname in cluster_names.items()}


def create_rosling_chart_arrows(df, arrow_opacity=0.5, arrow_width=2):
    """
    Hans Rosling-style chart with arrows from 1995 origin to current year.
    """
    data = df.copy()

    # Preprocessing
    data['Log GDP per capita'] = np.log(data[COL_GDP_PC])
    data['Production_Per_Capita'] = data[COL_PRODUCTION] / data['Population']

    # Fix cluster colors to 1995 values
    cluster_1995 = data[data[COL_YEAR] == 1995][[COL_COUNTRY_CODE, COL_CLUSTER]].copy()
    cluster_1995 = cluster_1995.rename(columns={COL_CLUSTER: 'Cluster_1995'})
    data = data.merge(cluster_1995, on=COL_COUNTRY_CODE, how='left')
    data = data.dropna(subset=['Cluster_1995', 'Log GDP per capita', COL_ECI, 'Production_Per_Capita'])

    # Convert string label -> int cluster id
    label_to_num = {v: k for k, v in cluster_names.items()}
    data['Cluster_1995'] = data['Cluster_1995'].map(label_to_num)
    data = data.dropna(subset=['Cluster_1995'])
    data['Cluster_1995'] = data['Cluster_1995'].astype(int)

    # Bubble sizing
    data['Bubble_Size'] = np.sqrt(data['Production_Per_Capita'])
    min_s, max_s = data['Bubble_Size'].min(), data['Bubble_Size'].max()
    data['Bubble_Size_Scaled'] = 8 + (data['Bubble_Size'] - min_s) / (max_s - min_s) * 42

    data = data.sort_values([COL_YEAR, COL_COUNTRY_CODE])
    years = sorted(data[COL_YEAR].unique())
    countries = data[COL_COUNTRY_CODE].unique()
    clusters = sorted(data['Cluster_1995'].unique())

    # Build country data dictionary
    country_data = {}
    for code in countries:
        cdf = data[data[COL_COUNTRY_CODE] == code].sort_values(COL_YEAR)

        origin_row = cdf[cdf[COL_YEAR] == 1995]
        if len(origin_row) == 0:
            continue

        country_data[code] = {
            'years': cdf[COL_YEAR].values,
            'x': cdf['Log GDP per capita'].values,
            'y': cdf[COL_ECI].values,
            'x_origin': origin_row['Log GDP per capita'].values[0],
            'y_origin': origin_row[COL_ECI].values[0],
            'size': cdf['Bubble_Size_Scaled'].values,
            'name': cdf[COL_COUNTRY_NAME].iloc[0],
            'cluster': cdf['Cluster_1995'].iloc[0],
            'prod_pc': cdf['Production_Per_Capita'].values
        }

    countries = list(country_data.keys())

    # ============ BUILD FIGURE ============
    fig = go.Figure()

    first_year = years[0]

    for cluster in clusters:
        cluster_countries = [c for c in countries if country_data[c]['cluster'] == cluster]
        color = CLUSTER_COLORS[cluster]

        # Add ARROW traces
        for code in cluster_countries:
            cd = country_data[code]
            idx = np.where(cd['years'] == first_year)[0]

            if len(idx) > 0:
                i = idx[0]
                x_current, y_current = cd['x'][i], cd['y'][i]
            else:
                x_current, y_current = cd['x_origin'], cd['y_origin']

            fig.add_trace(go.Scatter(
                x=[cd['x_origin'], x_current],
                y=[cd['y_origin'], y_current],
                mode='lines',
                line=dict(color=color, width=arrow_width),
                opacity=arrow_opacity,
                legendgroup=f"cluster_{cluster}",
                showlegend=False,
                hoverinfo='skip'
            ))

        # Add BUBBLE traces
        for code in cluster_countries:
            cd = country_data[code]
            idx = np.where(cd['years'] == first_year)[0]

            if len(idx) > 0:
                i = idx[0]
                x_val, y_val = [cd['x'][i]], [cd['y'][i]]
                size_val, prod_val = cd['size'][i], cd['prod_pc'][i]
            else:
                x_val, y_val = [cd['x_origin']], [cd['y_origin']]
                size_val, prod_val = 15, 0

            fig.add_trace(go.Scatter(
                x=x_val, y=y_val,
                mode='markers+text',
                marker=dict(size=size_val, color=color, opacity=0.85,
                            line=dict(width=1.5, color='white')),
                text=[code],
                textposition='top center',
                textfont=dict(size=8, color='black'),
                name=CLUSTER_NAMES[cluster],
                legendgroup=f"cluster_{cluster}",
                showlegend=(code == cluster_countries[0]),
                customdata=[[cd['name'], prod_val, first_year]],
                hovertemplate=(
                    "<b>%{customdata[0]}</b><br>"
                    "Log GDP pc: %{x:.2f}<br>"
                    "ECI: %{y:.2f}<br>"
                    "Prod/capita: $%{customdata[1]:,.0f}<br>"
                    "Year: %{customdata[2]}<extra></extra>"
                )
            ))

        # Add ORIGIN MARKERS
        for code in cluster_countries:
            cd = country_data[code]
            fig.add_trace(go.Scatter(
                x=[cd['x_origin']],
                y=[cd['y_origin']],
                mode='markers',
                marker=dict(size=5, color=color, opacity=0.6, symbol='circle'),
                legendgroup=f"cluster_{cluster}",
                showlegend=False,
                hoverinfo='skip'
            ))

    # ============ BUILD FRAMES ============
    frames = []
    for year in years:
        frame_data = []

        for cluster in clusters:
            cluster_countries = [c for c in countries if country_data[c]['cluster'] == cluster]
            color = CLUSTER_COLORS[cluster]

            for code in cluster_countries:
                cd = country_data[code]
                idx = np.where(cd['years'] == year)[0]

                if len(idx) > 0:
                    i = idx[0]
                    x_current, y_current = cd['x'][i], cd['y'][i]
                else:
                    valid_mask = cd['years'] <= year
                    if valid_mask.any():
                        last_idx = np.where(valid_mask)[0][-1]
                        x_current, y_current = cd['x'][last_idx], cd['y'][last_idx]
                    else:
                        x_current, y_current = cd['x_origin'], cd['y_origin']

                frame_data.append(go.Scatter(
                    x=[cd['x_origin'], x_current],
                    y=[cd['y_origin'], y_current],
                    mode='lines',
                    line=dict(color=color, width=arrow_width),
                    opacity=arrow_opacity
                ))

            for code in cluster_countries:
                cd = country_data[code]
                idx = np.where(cd['years'] == year)[0]

                if len(idx) > 0:
                    i = idx[0]
                    x_val, y_val = [cd['x'][i]], [cd['y'][i]]
                    size_val, prod_val = cd['size'][i], cd['prod_pc'][i]
                else:
                    valid_mask = cd['years'] <= year
                    if valid_mask.any():
                        last_idx = np.where(valid_mask)[0][-1]
                        x_val, y_val = [cd['x'][last_idx]], [cd['y'][last_idx]]
                        size_val, prod_val = cd['size'][last_idx], cd['prod_pc'][last_idx]
                    else:
                        x_val, y_val = [cd['x_origin']], [cd['y_origin']]
                        size_val, prod_val = 15, 0

                frame_data.append(go.Scatter(
                    x=x_val, y=y_val,
                    mode='markers+text',
                    marker=dict(size=size_val, color=color, opacity=0.85,
                                line=dict(width=1.5, color='white')),
                    text=[code],
                    textposition='top center',
                    textfont=dict(size=8, color='black'),
                    customdata=[[cd['name'], prod_val, year]],
                    hovertemplate=(
                        "<b>%{customdata[0]}</b><br>"
                        "Log GDP pc: %{x:.2f}<br>"
                        "ECI: %{y:.2f}<br>"
                        "Prod/capita: $%{customdata[1]:,.0f}<br>"
                        "Year: %{customdata[2]}<extra></extra>"
                    )
                ))

            for code in cluster_countries:
                cd = country_data[code]
                frame_data.append(go.Scatter(
                    x=[cd['x_origin']],
                    y=[cd['y_origin']],
                    mode='markers',
                    marker=dict(size=5, color=color, opacity=0.6, symbol='circle')
                ))

        frames.append(go.Frame(data=frame_data, name=str(year)))

    fig.frames = frames

    # ============ LAYOUT ============
    for eci_val in [-1, 0, 1]:
        fig.add_hline(y=eci_val, line_dash="dot",
                      line_color="rgba(150,150,150,0.4)", line_width=1)

    eci_min, eci_max = data[COL_ECI].min(), data[COL_ECI].max()
    x_min, x_max = data['Log GDP per capita'].min(), data['Log GDP per capita'].max()

    fig.update_layout(
        title=dict(
            text='Evolution of Economic Complexity vs Income<br>'
                 '<sup> Bubble size = Production per Capita</sup>',
            x=0.5, xanchor='center'
        ),
        xaxis=dict(range=[x_min - 0.2, x_max + 0.2],
                   title='Log GDP per capita (PPP)',
                   gridcolor='rgba(200,200,200,0.3)', showgrid=True),
        yaxis=dict(range=[eci_min - 0.5, eci_max + 0.5],
                   title='Economic Complexity Index',
                   gridcolor='rgba(200,200,200,0.3)', showgrid=True),
        plot_bgcolor='white',
        legend=dict(title='Resource Profile (1995)', x=1.02, y=0.99),
        width=850, height=650,
        updatemenus=[
            dict(type='buttons', showactive=True, x=1.0, y=-0.02,
                 xanchor='left', yanchor='top',
                 buttons=[
                     dict(label='▶', method='animate',
                          args=[None, dict(frame=dict(duration=500, redraw=True),
                                           fromcurrent=True,
                                           transition=dict(duration=300))]),
                     dict(label='⏸', method='animate',
                          args=[[None], dict(frame=dict(duration=0), mode='immediate')])
                 ])
        ],
        sliders=[{
            'active': 0,
            'currentvalue': {'prefix': 'Year: ', 'font': {'size': 14}, 'xanchor': 'left', 'offset': 10},
            'len': 0.85, 'x': 0.05, 'y': -0.12,
            'pad': {'t': 30},
            'steps': [dict(args=[[str(y)], dict(frame=dict(duration=300, redraw=True),
                                                mode='immediate')],
                           method='animate', label=str(y)) for y in years]
        }]
    )

    return fig


# ============ RUN ============
fig = create_rosling_chart_arrows(master, arrow_opacity=0.5, arrow_width=2)
fig.write_image("..\\figures\\clusters\\rosling_arrows.png")
fig.show()

fig.write_html(r"..\figures\clusters\rosling_arrows.html")